## 906 LVTestCase — GNN dataset generation (chunked)

Generate chunked OpenDSS snapshot data for DA-GPS training on the **IEEE European LV Test Case** (906 buses / LVTestCase).

This mirrors the 8500 **no-BESS** chunked driver in `withder.ipynb` (calls a thin Python generator per chunk).

### Full vs smoke sizes (same as 8500 chunked)
| Mode | `TOTAL_SCENARIOS` | `N_SAMPLES_PER_SCENARIO` | `SCENARIOS_PER_CHUNK` | chunks |
|---|---|---|---|---|
| Full (`SMOKE_TEST=False`) | 2000 | 40 | 50 | 40 |
| Smoke (`SMOKE_TEST=True`) | 4 | 8 | 2 | 2 |

### Save location (`CHUNK_ROOT`)
- **Colab:** `/content/drive/MyDrive/datasets_gnn2/original_906_lvtestcase_chunked`
- **Windows (preferred):** `K:\My Drive\datasets_gnn2\original_906_lvtestcase_chunked`
- Fallback: `D:\datasets\...` or `{repo}\datasets_gnn2\...`

### Differences vs 8500
| | IEEE 8500 unbalanced | 906 LVTestCase |
|---|---|---|
| DER | PV (+ optional BESS) | none (`p_pv_kw=0`) |
| Controls | regs + caps (real meta) | **none** — dummy constant 8500 `reg_*` / `cap_*` meta cols for trainer schema |
| Profiles | 5-min load + irradiance (288 pts) | 1-min per-load shapes (1440 pts) |
| MV aggregation | split-phase → MV `*_mvagg.csv` | **no real mvagg** — writes `gnn_node_features_and_targets_mvagg.csv` as a compat copy |
| Physics loss | optional (8500 Y / device catalog) | keep **off** (no regs/caps / no 8500 catalog) |

### Per-chunk outputs (`run_***/`)
- `gnn_node_index_master.csv` (incl. Laplacian PE)
- `gnn_edges_phase_static.csv`
- `gnn_sample_meta.csv` (incl. dummy cap/reg columns)
- `gnn_node_features_and_targets_mvagg.csv` ← what DA-GPS loads

### After generation
Point the DA-GPS training cell’s `CHUNK_PARENT` at `CHUNK_ROOT` below (same layout as 8500: parent of `run_*` folders). See the short follow-up cell after the generator.

In [ ]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario — 906 LVTestCase
# - runs in chunks
# - different seed per chunk
# - different output folder per chunk
# - PE computed from static edges each chunk (node_pe_k > 0, no CSV)
# - writes *_mvagg.csv compat file (identity; no split-phase aggregation)
# - dummy 8500 cap/reg meta columns for existing DA-GPS trainer schema
# ============================================================
import os
import math
import time
from pathlib import Path

# ---------------- user controls ----------------
SMOKE_TEST = False  # True = tiny run to verify paths / OpenDSS; False = full dataset

TOTAL_SCENARIOS = 2000
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 90620230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.0  # unused (no PV); kept for parity with 8500 cell

# Nominal: 55 x 1 kW @ PF 0.95  →  P≈55 kW, Q≈18.08 kvar
P_LOAD_MEAN_KW = 55.0
Q_LOAD_MEAN_KVAR = 18.077625784837476
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.0, 0.0)  # no PV on LVTestCase

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False
WRITE_MVAGG_COMPAT = True
DELETE_RAW_NODE_CSV_AFTER_MVAGG = True  # training only needs *_mvagg.csv

if SMOKE_TEST:
    TOTAL_SCENARIOS = 4
    N_SAMPLES_PER_SCENARIO = 8
    SCENARIOS_PER_CHUNK = 2
    print("SMOKE_TEST=True → TOTAL_SCENARIOS=4, N_SAMPLES_PER_SCENARIO=8, SCENARIOS_PER_CHUNK=2")

# Colab / Windows / local defaults — edit CHUNK_ROOT to your Drive / disk
CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    "/content/GNN2",
    os.getcwd(),
]

_REPO = None
for root in CANDIDATES:
    if os.path.isdir(root) and os.path.isfile(os.path.join(root, "run_original_style_dataset_906_lvtestcase.py")):
        _REPO = os.path.abspath(root)
        break
if _REPO is None:
    raise FileNotFoundError("Could not locate repo root containing run_original_style_dataset_906_lvtestcase.py")

# Do NOT use Path("/content/...").exists() — on Windows that is drive-root-relative
# (e.g. C:\content\...) and stays True forever after a mistaken mkdir.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_K_GNN2 = Path(r"K:\My Drive\datasets_gnn2")
_K_MYDRIVE = Path(r"K:\My Drive")

if IN_COLAB:
    CHUNK_ROOT = Path("/content/drive/MyDrive/datasets_gnn2/original_906_lvtestcase_chunked")
elif _K_GNN2.exists() or _K_MYDRIVE.exists():
    CHUNK_ROOT = _K_GNN2 / "original_906_lvtestcase_chunked"
elif Path(r"D:\datasets").exists():
    CHUNK_ROOT = Path(r"D:\datasets\original_906_lvtestcase_chunked")
else:
    CHUNK_ROOT = Path(_REPO) / "datasets_gnn2" / "original_906_lvtestcase_chunked"

CHUNK_ROOT.mkdir(parents=True, exist_ok=True)
print("CHUNK_ROOT:", CHUNK_ROOT)

script_path = os.path.join(_REPO, "run_original_style_dataset_906_lvtestcase.py")
os.chdir(_REPO)
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)

ns = {"__name__": "run_original_style_dataset_906_lvtestcase", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

# ---------------- preflight ----------------
model_dir = ns["MODEL_DIR"]
master_dss = ns["MASTER_DSS"]
print("MODEL_DIR:", model_dir)
print("MASTER_DSS exists:", master_dss.is_file())
if not master_dss.is_file():
    raise FileNotFoundError(f"Missing LVTestCase Master.dss: {master_dss}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | SMOKE_TEST={SMOKE_TEST}")
print(f"P_LOAD_MEAN_KW={P_LOAD_MEAN_KW}  Q_LOAD_MEAN_KVAR={Q_LOAD_MEAN_KVAR}")

for chunk_idx in range(n_chunks):
    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"
    ns["MVAGG_CSV"] = out_dir / "gnn_node_features_and_targets_mvagg.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=False,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        write_mvagg_compat=bool(WRITE_MVAGG_COMPAT),
        delete_raw_node_csv_after_mvagg=bool(DELETE_RAW_NODE_CSV_AFTER_MVAGG),
    )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_906_lvtestcase"](**gen_kw)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    print("Saved:")
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    if not DELETE_RAW_NODE_CSV_AFTER_MVAGG:
        print(" -", ns["NODE_CSV"])
    print(" -", ns["MVAGG_CSV"])

print("\nAll chunks finished.")
print("Next: set DA-GPS CHUNK_PARENT to:", CHUNK_ROOT)


## DA-GPS training on 906 LVTestCase (chunked)

Train the existing multitask DA-GPS model on the **already-generated** 906 chunked dataset. **Do not regenerate** data for training — point at:

- Local: `K:\My Drive\datasets_gnn2\original_906_lvtestcase_chunked`
- Colab: `/content/drive/MyDrive/datasets_gnn2/original_906_lvtestcase_chunked`

### System / latent tokens (this trainer)

`--n_system_tokens` allocates learnable **global tokens** after the fixed cap-bank and regulator tokens. With `--aux_meta_cols col0,col1,...`, column `i` supervises system-token slot `i` (global index `n_cap + n_reg + i`) via normalized MSE (`--lambda_pv`). Remaining system tokens (if any) stay unsupervised latents. For 906 we use **2** tokens tied to **substation upstream P and Q**: `P_grid_upstream_post_kw,Q_grid_upstream_post_kvar`.

### 906 vs 8500 training differences

| Setting | 8500 (typical) | 906 (this cell) |
|--------|----------------|-----------------|
| Dataset | `original_8500_..._2000_40` | `original_906_lvtestcase_chunked` |
| Meta aux | PV + losses | Substation P/Q only |
| `n_system_tokens` | 10 | 2 |
| Cap / reg loss | `lambda_cap/reg > 0` | **0** (dummy columns only) |
| PV aux weight | `lambda_pv > 0` on PV cols | `lambda_pv` on grid P/Q (no `pv_pv2_*`) |
| Physics | optional PF catalogs | **always off** (`PHYSICS_WEIGHT=0`) |
| Loss meta units | as stored | `--meta_loss_scale 0.001` if/when `P_loss_*` / `Q_loss_*` are ingested |

### Loss scale without regenerating CSVs

906 meta currently stores `P_loss_total_post_kw` / `Q_loss_total_post_kvar` ~**1000x** too large. The trainer flag `--meta_loss_scale 0.001` multiplies those two columns whenever they are loaded into aux/cache targets (default `1.0` keeps 8500 unchanged). This cell passes `0.001` for correctness if losses are ever added to `--aux_meta_cols`; substation P/Q tokens are **not** scaled.

### Smoke vs full

- `SMOKE_TEST=True`: prefer the first few **full-size** `run_*` chunks (scen span ≥ 50 in the folder name); if none exist, fall back to any `run_*`. Fewer epochs (fast check).
- `SMOKE_TEST=False`: **all full-size** chunks only (scen span ≥ 50). Smoke folders like `run_001_scen_0000_0001_...` (span=2) are **auto-skipped** — leave them on disk; no need to delete.

Caches use **906-specific** names (`da_gps_chunked_906_...`) so they never collide with 8500 caches. Runs go under `.../datasets_gnn2/runs`.


In [ ]:
# --- DA-GPS training on 906 LVTestCase (long-running; Colab GPU or local CUDA) ---
import os
import sys
import re
import subprocess
import datetime
import warnings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / "original_906_lvtestcase_chunked"
WIN_CHUNK_DEFAULT = Path(r"K:\My Drive\datasets_gnn2\original_906_lvtestcase_chunked")
WIN_RUNS_DEFAULT = Path(r"K:\My Drive\datasets_gnn2\runs")
WIN_CACHE_PARENT = Path(r"K:\My Drive\datasets_gnn2\cache")


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    """Resolve a data path; never join Windows drive letters with cwd on Linux."""
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(
            f"No run_*/gnn_node_index_master.csv under {chunk_parent}"
        )
    return hits[0]


def _sorted_run_chunk_dirs(chunk_parent: Path) -> list[Path]:
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


_FULL_CHUNK_MIN_SCEN = 50  # full: scen_0000_0049 => 50; smoke: scen_0000_0001 => 2


def _parse_run_scen_span(name: str) -> int | None:
    """Return (end - start + 1) from run_*_scen_{start}_{end}_seed_*, else None."""
    m = re.search(r"_scen_(\d+)_(\d+)(?:_|$)", name)
    if not m:
        return None
    start, end = int(m.group(1)), int(m.group(2))
    if end < start:
        return None
    return end - start + 1


def _classify_run_chunks(chunk_parent: Path, *, min_full_span: int = _FULL_CHUNK_MIN_SCEN):
    """Split run_* dirs into full vs other/smoke by scen span in the folder name."""
    all_runs = _sorted_run_chunk_dirs(chunk_parent)
    full, other = [], []
    for p in all_runs:
        span = _parse_run_scen_span(p.name)
        if span is not None and span >= min_full_span:
            full.append(p)
        else:
            other.append((p, span))
    return all_runs, full, other


def _select_training_chunk_glob(
    chunk_parent: Path,
    *,
    smoke: bool,
    smoke_count: int,
    min_full_span: int = _FULL_CHUNK_MIN_SCEN,
) -> str:
    """Build --chunk_subdir_glob with full vs smoke awareness.

    Full training (smoke=False): keep only folders whose scen span >= min_full_span
    (typically 50). Smoke folders (span=2) are skipped without deleting them.

    Smoke training (smoke=True): prefer the first `smoke_count` *full-size* chunks
    when available; otherwise fall back to the first `smoke_count` any run_*.
    """
    all_runs, full, other = _classify_run_chunks(chunk_parent, min_full_span=min_full_span)
    if not all_runs:
        raise FileNotFoundError(f"No run_* under {chunk_parent}")

    print(
        f"Chunk selection (SMOKE_TEST={smoke}, min_full_scen_span={min_full_span}): "
        f"{len(all_runs)} run_* found; {len(full)} full-size, {len(other)} other/smoke"
    )

    if smoke:
        pool = full if full else all_runs
        source = "full-size" if full else "any run_* (no full-size chunks found)"
        if len(pool) < smoke_count:
            raise ValueError(
                f"SMOKE_CHUNK_COUNT={smoke_count} but only {len(pool)} {source} "
                f"under {chunk_parent}"
            )
        kept = pool[:smoke_count]
        kept_names = {p.name for p in kept}
        print(f"  smoke: taking first {smoke_count} from {source}")
        for p in all_runs:
            span = _parse_run_scen_span(p.name)
            span_s = f"span={span}" if span is not None else "unparseable span"
            if p.name in kept_names:
                print(f"  KEEP  {p.name} ({span_s})")
            elif full and p not in full:
                print(f"  SKIP  {p.name} ({span_s}; smoke prefers full-size chunks)")
            else:
                print(f"  SKIP  {p.name} ({span_s}; beyond first {smoke_count})")
        return ",".join(p.name for p in kept)

    if not full:
        raise RuntimeError(
            f"SMOKE_TEST=False but no full-size chunks (scen span >= {min_full_span}) "
            f"under {chunk_parent}. Found {len(all_runs)} run_* folder(s)."
        )
    for p, span in other:
        span_s = f"span={span}" if span is not None else "unparseable span"
        print(
            f"  SKIP  {p.name} ({span_s}; need span>={min_full_span} for full training)"
        )
    for p in full:
        span = _parse_run_scen_span(p.name)
        print(f"  KEEP  {p.name} (span={span}; full)")
    return ",".join(p.name for p in full)


def _chunks_from_subdir_glob(chunk_parent: Path, glob_pat: str) -> list[Path]:
    import fnmatch

    glob_pat = str(glob_pat).strip()
    if "," in glob_pat:
        allowed = {s.strip() for s in glob_pat.split(",") if s.strip()}
        chunks = sorted(
            (p for p in chunk_parent.iterdir() if p.is_dir() and p.name in allowed),
            key=lambda p: p.name,
        )
        missing = allowed - {p.name for p in chunks}
        if missing:
            raise FileNotFoundError(f"Missing smoke chunk folders: {sorted(missing)}")
        return chunks
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and fnmatch.fnmatch(p.name, glob_pat)),
        key=lambda p: p.name,
    )


def _write_smoke_compare_manifest(
    runs_parent: Path,
    *,
    out_dir: Path,
    physics_weight: float,
    chunk_glob: str,
    n_chunks: int,
    epochs: int,
    seed: int,
) -> None:
    import json as _json

    path = runs_parent / "_smoke_compare_manifest_906.json"
    manifest = {}
    if path.is_file():
        manifest = _json.loads(path.read_text(encoding="utf-8"))
    key = "physics" if physics_weight > 0 else "baseline"
    manifest[key] = {
        "run_dir": str(out_dir.resolve()),
        "physics_weight": physics_weight,
        "chunk_glob": chunk_glob,
        "n_chunks": n_chunks,
        "epochs": epochs,
        "seed": seed,
        "feeder": "906_lvtestcase",
    }
    path.write_text(_json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Smoke compare manifest ({key}) -> {path}")


# Colab: mount Drive when chunks/caches live under MyDrive
if _on_colab() and not _drive_mounted():
    from google.colab import drive
    drive.mount("/content/drive")

# --- smoke / full-run toggles ---
SMOKE_TEST = True
SMOKE_CHUNK_COUNT = 3  # first N full-size run_* when available; 3-5 = fast multi-chunk
SMOKE_EPOCHS = 15
SMOKE_PATIENCE = 5
SMOKE_SEED = 42
_DA_CACHE_NAME = "da_gps_chunked_906_smoke_gine" if SMOKE_TEST else "da_gps_chunked_906_full_gine"
_GNN_CACHE_NAME = "gnn_only_chunked_906_full_gine"

# --- paths: auto-detect Colab + Drive; prefer K: on Windows ---
_env_repo = os.environ.get("GNN2_REPO_ROOT", "").strip()
REPO = Path(_env_repo).expanduser().resolve() if _env_repo else Path.cwd().resolve()
if not (REPO / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
    raise FileNotFoundError(f"Repo root missing trainer script: {REPO}")

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError(
            "Colab requires Google Drive mounted. Re-run this cell and approve Drive access, "
            "or: from google.colab import drive; drive.mount('/content/drive')"
        )
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_GNN_CACHE_NAME}"
    RUNS_PARENT = MYDRIVE_DATA / "runs"
elif os.name == "nt":
    CHUNK_PARENT = WIN_CHUNK_DEFAULT
    DA_CACHE_ROOT = WIN_CACHE_PARENT / _DA_CACHE_NAME
    GNN_CACHE_ROOT = WIN_CACHE_PARENT / _GNN_CACHE_NAME
    RUNS_PARENT = WIN_RUNS_DEFAULT
else:
    CHUNK_PARENT = REPO / "datasets_gnn2_from pc/original_906_lvtestcase_chunked"
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_GNN_CACHE_NAME}"
    RUNS_PARENT = REPO / "datasets_gnn2_from pc/runs"

NODE_PE_CSV = None  # auto: first run_*/gnn_node_index_master.csv under CHUNK_PARENT

# Optional overrides (use POSIX paths on Colab, not K:\...):
# CHUNK_PARENT = Path("/content/drive/MyDrive/datasets_gnn2/original_906_lvtestcase_chunked")
# NODE_PE_CSV = CHUNK_PARENT / "run_001_scen_0000_0001_seed_90720233/gnn_node_index_master.csv"

# --- lighter architecture ---
LIGHTER_HIDDEN = 64
LIGHTER_LAYERS = 2
LIGHTER_HEADS = 2
LIGHTER_NODE_EMB_DIM = 2

PHYSICS_WEIGHT = 0.0  # always off for 906 (no 8500 PF catalogs)

# Substation upstream P/Q only (no PV aux, no loss tokens)
META_AUX_COLS = "P_grid_upstream_post_kw,Q_grid_upstream_post_kvar"
N_SYSTEM_TOKENS = 2
META_LOSS_SCALE = 0.001  # fix buggy stored P/Q loss units if those cols are ingested
LAMBDA_PV = 0.1  # weight for meta-aux MSE on the 2 system tokens
LAMBDA_CAP = 0.0
LAMBDA_REG = 0.0

NUM_WORKERS = 0 if os.name == "nt" else 4

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
else:
    EPOCHS = 200
    PATIENCE = 30
# Full: only scen-span >= 50 folders. Smoke: prefer first N full-size, else any run_*.
CHUNK_GLOB = _select_training_chunk_glob(
    chunk_parent, smoke=SMOKE_TEST, smoke_count=SMOKE_CHUNK_COUNT
)
da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_GNN_CACHE_NAME}" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

print("=== Preflight (906 DA-GPS) ===")
print(f"REPO:           {REPO}")
print(f"SMOKE_TEST:     {SMOKE_TEST}")
print(f"SMOKE_CHUNK_COUNT: {SMOKE_CHUNK_COUNT}")
print(f"CHUNK_GLOB:     {CHUNK_GLOB}")
print(f"EPOCHS:         {EPOCHS}")
print(f"PATIENCE:       {PATIENCE}")
print(f"CHUNK_PARENT:   {chunk_parent}")
print(f"META_AUX_COLS:  {META_AUX_COLS}")
print(f"N_SYSTEM_TOKENS:{N_SYSTEM_TOKENS}")
print(f"META_LOSS_SCALE:{META_LOSS_SCALE}")
print(f"lambda_cap/reg/pv: {LAMBDA_CAP}/{LAMBDA_REG}/{LAMBDA_PV}")
print(f"PHYSICS_WEIGHT: {PHYSICS_WEIGHT}")
if not chunk_parent.is_dir():
    raise FileNotFoundError(f"CHUNK_PARENT not found: {chunk_parent}")

run_preview = sorted(
    p.name for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")
)[:5]
print(f"run_* preview:  {run_preview}")

if NODE_PE_CSV is None:
    node_pe = _find_node_pe_csv(chunk_parent)
else:
    node_pe = _resolve_data_path(NODE_PE_CSV, label="NODE_PE_CSV", colab_fallback=None)
    if not node_pe.is_file():
        warnings.warn(
            f"NODE_PE_CSV not found at {node_pe}; auto-discovering under CHUNK_PARENT.",
            UserWarning,
            stacklevel=2,
        )
        node_pe = _find_node_pe_csv(chunk_parent)

print(f"NODE_PE_CSV:    {node_pe}")
print(f"DA_CACHE_ROOT:  {da_cache_root}")
print(f"GNN_CACHE_ROOT: {gnn_cache_root}")
print(f"RUNS_PARENT:    {runs_parent}")
print("PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)")
print("=================")

da_cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

chunks = _chunks_from_subdir_glob(chunk_parent, CHUNK_GLOB)
if not chunks:
    raise RuntimeError(f"No folders for CHUNK_GLOB={CHUNK_GLOB!r} under {chunk_parent}")

for p in chunks:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p.name}")

print(f"Found {len(chunks)} chunk(s) for CHUNK_GLOB under {chunk_parent}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = runs_parent / (
    f"da_gps_906_l{LIGHTER_LAYERS}_h{LIGHTER_HIDDEN}_gine_gridPQ"
    f"{'_smoke' if SMOKE_TEST else ''}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

_early_stop = "total"

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", str(N_SYSTEM_TOKENS),
    "--aux_meta_cols", META_AUX_COLS,
    "--meta_loss_scale", str(META_LOSS_SCALE),
    "--lambda_pv", str(LAMBDA_PV),
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", "64",
    "--hidden", str(LIGHTER_HIDDEN),
    "--layers", str(LIGHTER_LAYERS),
    "--heads", str(LIGHTER_HEADS),
    "--node_emb_dim", str(LIGHTER_NODE_EMB_DIM),
    "--edge_emb_dim", "0",
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", str(LAMBDA_CAP),
    "--lambda_reg", str(LAMBDA_REG),
    "--reg_loss", "mse",
    "--patience", str(PATIENCE),
    "--seed", str(SMOKE_SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--log_every", "10",
    "--checkpoint_every", "10",
    "--early_stop_on", _early_stop,
    "--dropout", "0.1",
]

# PHYSICS_WEIGHT must stay 0 for 906 — no PF flags added.

print(f"PHYSICS_WEIGHT={PHYSICS_WEIGHT} (baseline; no 8500 physics)")
print(f"early_stop_on={_early_stop}")
print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
_write_smoke_compare_manifest(
    runs_parent,
    out_dir=out_dir,
    physics_weight=PHYSICS_WEIGHT,
    chunk_glob=CHUNK_GLOB,
    n_chunks=len(chunks),
    epochs=EPOCHS,
    seed=SMOKE_SEED if SMOKE_TEST else 42,
)
